In [19]:
import pandas as pd

# Caminho do arquivo de entrada
arquivo =  r"C:\Users\Magda\Downloads\Cópia de Planilha de Case para KPIs.xlsx"
arquivo_saida = r"C:\Users\Magda\Downloads\resumo_criacao_passagem_ganho.xlsx"

# Leitura das abas
deal = pd.read_excel(arquivo, sheet_name='deal list')
divisao = pd.read_excel(arquivo, sheet_name='Divisão Shopping')

# Padroniza nomes
deal.columns = deal.columns.str.strip()
divisao.columns = divisao.columns.str.strip()

# Renomeia campos para facilitar merge
divisao = divisao.rename(columns={'Nome': 'Nome_Match', 'Origem': 'Loja'})

# Função para extrair AnoMes
def extrair_mes(df, col_data):
    df[col_data] = pd.to_datetime(df[col_data], errors='coerce')
    df['AnoMes'] = df[col_data].dt.to_period('M').astype(str)
    return df

# -------------------- TABELA DE CRIAÇÃO --------------------
df_criacao = deal[['Negócio - Data recebido', 'Negócio - Criado por']].dropna()
df_criacao = df_criacao.rename(columns={'Negócio - Criado por': 'Nome_Match'})
df_criacao = extrair_mes(df_criacao, 'Negócio - Data recebido')
df_criacao = df_criacao.merge(divisao, on='Nome_Match', how='left')
df_criacao = df_criacao.groupby(['AnoMes', 'Loja']).size().reset_index(name='Criacao')

# -------------------- TABELA DE PASSAGEM --------------------
df_passagem = deal[['Negócio - Data recebido', 'Negócio - [SDR] Original']].dropna()
df_passagem = df_passagem.rename(columns={'Negócio - [SDR] Original': 'Nome_Match'})
df_passagem = extrair_mes(df_passagem, 'Negócio - Data recebido')
df_passagem = df_passagem.merge(divisao, on='Nome_Match', how='left')
df_passagem = df_passagem.groupby(['AnoMes', 'Loja']).size().reset_index(name='Passagem')

# -------------------- TABELA DE GANHO --------------------
df_ganho = deal[
    (deal['Negócio - Data recebido'].notna()) &
    (deal['Negócio - Proprietário'].notna()) &
    (deal['Negócio - Status'].str.lower() == 'ganho')
][['Negócio - Data recebido', 'Negócio - Proprietário']]
df_ganho = df_ganho.rename(columns={'Negócio - Proprietário': 'Nome_Match'})
df_ganho = extrair_mes(df_ganho, 'Negócio - Data recebido')
df_ganho = df_ganho.merge(divisao, on='Nome_Match', how='left')
df_ganho = df_ganho.groupby(['AnoMes', 'Loja']).size().reset_index(name='Ganho')

# -------------------- JUNÇÃO FINAL --------------------
resumo = pd.merge(df_criacao, df_passagem, on=['AnoMes', 'Loja'], how='outer')
resumo = pd.merge(resumo, df_ganho, on=['AnoMes', 'Loja'], how='outer')
resumo = resumo.fillna(0).sort_values(['AnoMes', 'Loja'])

# Converte contagens para inteiros
resumo[['Criacao', 'Passagem', 'Ganho']] = resumo[['Criacao', 'Passagem', 'Ganho']].astype(int)

# Salva o resultado
resumo.to_excel(arquivo_saida, index=False)

print(f"Resumo criado com sucesso: '{arquivo_saida}'")


Resumo criado com sucesso: 'C:\Users\Magda\Downloads\resumo_criacao_passagem_ganho.xlsx'


In [20]:
resumo[resumo['AnoMes'] == '2025-01']


,AnoMes,Loja,Criacao,Passagem,Ganho
165,2025-01,Anália Franco,5,174,6
166,2025-01,Barra Shopping,7,76,14
167,2025-01,Barra Sul,1,67,23
168,2025-01,Blumenau,8,66,16
212,2025-01,Campinas,0,1,0
169,2025-01,Canoas,1,57,7
213,2025-01,Catuaí Londrina,0,130,13
214,2025-01,Catuaí Maringá,0,104,6
215,2025-01,Criciuma,0,50,22
216,2025-01,Joinville,0,91,12


In [22]:
pip install matplotlib python-pptx


In [23]:

import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches 
import os

# Caminhos
arquivo_resumo = r"C:\Users\Magda\Downloads\resumo_criacao_passagem_ganho.xlsx"
arquivo_ppt = r"C:\Users\Magda\Downloads\graficos_loja_mes.pptx"

# Lê o resumo
resumo = pd.read_excel(arquivo_resumo)

# Cria a apresentação
prs = Presentation()
title_slide_layout = prs.slide_layouts[5]  # Slide em branco

# Lista de lojas
lojas = resumo['Loja'].dropna().unique()

for loja in lojas:
    df_loja = resumo[resumo['Loja'] == loja].sort_values('AnoMes')

    plt.figure(figsize=(10, 4))
    plt.plot(df_loja['AnoMes'], df_loja['Criacao'], label='Criação', marker='o')
    plt.plot(df_loja['AnoMes'], df_loja['Passagem'], label='Passagem', marker='o')
    plt.plot(df_loja['AnoMes'], df_loja['Ganho'], label='Ganho', marker='o')
    
    plt.title(f'KPIs por Mês - {loja}')
    plt.xlabel('Ano/Mês')
    plt.ylabel('Quantidade')
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    # Salva a figura temporária
    img_path = f'temp_plot_{loja}.png'
    plt.savefig(img_path)
    plt.close()

    # Adiciona slide e imagem
    slide = prs.slides.add_slide(title_slide_layout)
    slide.shapes.add_picture(img_path, Inches(1), Inches(1), width=Inches(8), height=Inches(4.5))


    # Remove imagem temporária
    os.remove(img_path)

# Salva apresentação
prs.save(arquivo_ppt)
print(f"Gráficos salvos em: {arquivo_ppt}")


Gráficos salvos em: C:\Users\Magda\Downloads\graficos_loja_mes.pptx


In [3]:
import pandas as pd
# Caminho do arquivo original
arquivo = r"C:\Users\Magda\Downloads\Cópia de Planilha de Case para KPIs.xlsx"

# Leitura da aba
deal = pd.read_excel(arquivo, sheet_name='deal list')
deal.columns = deal.columns.str.strip()

# Conversão de datas (com dayfirst para formato d/m/yyyy)
deal['DataPassagem'] = pd.to_datetime(deal['Negócio - Data recebido'], errors='coerce')
deal['DataGanho'] = pd.to_datetime(deal['Negócio - Ganho em'], errors='coerce', dayfirst=True)


# Total de passagens
total_passagens = deal['DataPassagem'].notna().sum()

# Filtra ganhos com datas válidas
ganhos = deal[
    (deal['Negócio - Status'].str.lower() == 'ganho') &
    deal['DataPassagem'].notna() &
    deal['DataGanho'].notna()
].copy()

# Calcula diferença de dias entre passagem e ganho
ganhos['DiasParaGanho'] = (ganhos['DataGanho'] - ganhos['DataPassagem']).dt.days

# Calcula taxas de conversão por bucket
buckets = [1, 3, 7, 15]
resultados = []

for dias in buckets:
    ganhos_ate_dias = (ganhos['DiasParaGanho'] <= dias).sum()
    taxa = (ganhos_ate_dias / total_passagens) * 100
    resultados.append({
        'Prazo (dias)': dias,
        'Ganhos até o prazo': ganhos_ate_dias,
        'Total de passagens': total_passagens,
        'Taxa de conversão (%)': round(taxa, 2)
    })

# Converte em DataFrame e salva
df_resultado = pd.DataFrame(resultados)
df_resultado.to_excel(r"C:\Users\Magda\Downloads\taxa_conversao_geral.xlsx", index=False)

print("Resumo geral de conversão salvo com sucesso!")


Resumo geral de conversão salvo com sucesso!


In [13]:
#import pandas as pd

# Conversão de datas
deal['DataPassagem'] = pd.to_datetime(deal['Negócio - Data recebido'], errors='coerce')
deal['DataGanho'] = pd.to_datetime(deal['Negócio - Ganho em'], errors='coerce', dayfirst=True)


# Filtra apenas registros com passagem e status de ganho
deal_filtrado = deal[
    deal['DataPassagem'].notna()
].copy()

# Agrupa por dia da passagem
resultados = []

buckets = [1, 3, 7, 15]
dias_unicos = deal_filtrado['DataPassagem'].dt.date.unique()
dias_unicos.sort()

for dia in dias_unicos:
    subset = deal_filtrado[deal_filtrado['DataPassagem'].dt.date == dia].copy()
    total_passagens = len(subset)

    # Filtra os ganhos desse subset
    subset = subset[
        (subset['Negócio - Status'].str.lower() == 'ganho') &
        subset['DataGanho'].notna()
    ].copy()

    subset['DiasParaGanho'] = (subset['DataGanho'] - subset['DataPassagem']).dt.days

    for bucket in buckets:
        ganhos_ate_dias = (subset['DiasParaGanho'] <= bucket).sum()
        taxa = (ganhos_ate_dias / total_passagens) * 100 if total_passagens > 0 else 0
        resultados.append({
            'DataPassagem': dia,
            'Prazo (dias)': bucket,
            'Ganhos até o prazo': ganhos_ate_dias,
            'Total passagens do dia': total_passagens,
            'Taxa de conversão (%)': round(taxa, 2)
        })

# Cria DataFrame de saída
df_resultado = pd.DataFrame(resultados)

# Visualiza
print(df_resultado)

# Exporta se quiser
# df_resultado.to_excel("taxa_conversao_por_dia.xlsx", index=False)


     DataPassagem  Prazo (dias)  Ganhos até o prazo  Total passagens do dia  \
0      2022-12-09             1                   0                       1   
1      2022-12-09             3                   0                       1   
2      2022-12-09             7                   0                       1   
3      2022-12-09            15                   0                       1   
4      2023-11-08             1                   0                       1   
...           ...           ...                 ...                     ...   
1783   2025-03-18            15                   0                      12   
1784   2025-03-19             1                   0                       2   
1785   2025-03-19             3                   0                       2   
1786   2025-03-19             7                   0                       2   
1787   2025-03-19            15                   0                       2   

      Taxa de conversão (%)  
0                    

In [15]:



# Agrupa por data e bucket de prazo
agrupado = df_resultado.groupby(['DataPassagem', 'Prazo (dias)'], as_index=False).agg({
    'Ganhos até o prazo': 'sum',
    'Total passagens do dia': 'sum'
})

# Calcula nova taxa de conversão
agrupado['Taxa de conversão (%)'] = (
    agrupado['Ganhos até o prazo'] / agrupado['Total passagens do dia']
).fillna(0).round(2) * 100

# Visualiza os dados
print(agrupado)

# (Opcional) Exporta para Excel
# agrupado.to_excel("taxa_conversao_agrupada.xlsx", index=False)


     DataPassagem  Prazo (dias)  Ganhos até o prazo  Total passagens do dia  \
0      2022-12-09             1                   0                       1   
1      2022-12-09             3                   0                       1   
2      2022-12-09             7                   0                       1   
3      2022-12-09            15                   0                       1   
4      2023-11-08             1                   0                       1   
...           ...           ...                 ...                     ...   
1783   2025-03-18            15                   0                      12   
1784   2025-03-19             1                   0                       2   
1785   2025-03-19             3                   0                       2   
1786   2025-03-19             7                   0                       2   
1787   2025-03-19            15                   0                       2   

      Taxa de conversão (%)  
0                    

In [16]:
import matplotlib.pyplot as plt
import pandas as pd
from pptx import Presentation
from pptx.util import Inches
import os

# Supondo que 'agrupado' já está pronto com as colunas certas
agrupado['AnoMes'] = pd.to_datetime(agrupado['DataPassagem']).dt.to_period('M').astype(str)

# Agrupamento
grafico_df = agrupado.groupby(['AnoMes', 'Prazo (dias)'], as_index=False).agg({
    'Ganhos até o prazo': 'sum',
    'Total passagens do dia': 'sum'
})
grafico_df['Taxa de conversão (%)'] = (
    grafico_df['Ganhos até o prazo'] / grafico_df['Total passagens do dia']
).fillna(0).round(2) * 100

# Pivotar dados para gráfico empilhado
pivot_df = grafico_df.pivot(index='AnoMes', columns='Prazo (dias)', values='Taxa de conversão (%)').fillna(0)
pivot_df = pivot_df.sort_index()

# Criar gráfico de barras empilhadas
plt.figure(figsize=(12, 6))
bottom = pd.Series([0] * len(pivot_df), index=pivot_df.index)
colors = plt.cm.viridis([0.2, 0.4, 0.6, 0.8])  # Cores para 1,3,7,15 dias

for idx, prazo in enumerate(sorted(pivot_df.columns)):
    bars = plt.bar(pivot_df.index, pivot_df[prazo], bottom=bottom, label=f'{prazo} dias', color=colors[idx])
    bottom += pivot_df[prazo]
    
    # Adiciona valores dentro das barras
    for bar in bars:
        yval = bar.get_height()
        if yval > 0:  # Adiciona apenas se o valor for maior que 0
            plt.text(
                bar.get_x() + bar.get_width() / 2, 
                bar.get_height() / 2 + bar.get_y(), 
                f'{yval:.2f}%', 
                ha='center', 
                va='center',
                color='white', 
                fontsize=10
            )

# Adicionando título e labels
plt.title('Taxa de Conversão por Mês e Prazo')
plt.xlabel('Ano-Mês')
plt.ylabel('Taxa de Conversão (%)')
plt.xticks(rotation=45)
plt.legend(title='Prazo')
plt.tight_layout()

# Salvar imagem
img_path = 'grafico_empilhado_taxa_conversao_com_valores.png'
plt.savefig(img_path)
plt.close()

# Criar PowerPoint e adicionar slide
prs = Presentation()
blank_slide_layout = prs.slide_layouts[5]
slide = prs.slides.add_slide(blank_slide_layout)
slide.shapes.add_picture(img_path, Inches(1), Inches(1), height=Inches(4.5))

# Salvar apresentação
ppt_path = 'grafico_empilhado_conversão.pptx'
prs.save(ppt_path)

# Remover imagem temporária
os.remove(img_path)

print(f"Gráfico empilhado com valores salvo com sucesso em: {ppt_path}")


Gráfico empilhado com valores salvo com sucesso em: grafico_empilhado_conversão.pptx


In [30]:
import pandas as pd
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches
import os

# Leitura do arquivo
arquivo = r"C:\Users\Magda\Downloads\Cópia de Planilha de Case para KPIs.xlsx"
df = pd.read_excel(arquivo, sheet_name='deal list')
df.columns = df.columns.str.strip()

# Converte datas
df['Data_Recebido'] = pd.to_datetime(df['Negócio - Data recebido'], errors='coerce')
df['Data_Ganho'] = pd.to_datetime(df['Negócio - Ganho em'], dayfirst=True, errors='coerce')

# Filtra ganhos válidos
df = df[
    df['Data_Recebido'].notna() &
    df['Data_Ganho'].notna() &
    (df['Negócio - Status'].str.lower() == 'ganho')
].copy()

# Calcula tempo de conversão e mês de ganho
df['Dias_Conversao'] = (df['Data_Ganho'] - df['Data_Recebido']).dt.days
df['AnoMes'] = df['Data_Ganho'].dt.to_period('M').astype(str)

# Agrupa ganhos por mês
ganhos_total = df.groupby('AnoMes').size().rename("Ganho_Total")
ganhos_7 = df[df['Dias_Conversao'] <= 7].groupby('AnoMes').size().rename("Ganho_7d")
ganhos_15 = df[df['Dias_Conversao'] <= 15].groupby('AnoMes').size().rename("Ganho_15d")
ganhos_30 = df[df['Dias_Conversao'] <= 30].groupby('AnoMes').size().rename("Ganho_30d")

# Junta tudo
df_taxas = pd.concat([ganhos_total, ganhos_7, ganhos_15, ganhos_30], axis=1).fillna(0)

# Calcula taxas relativas ao total de ganhos
df_taxas['Taxa_7d'] = (df_taxas['Ganho_7d'] / df_taxas['Ganho_Total']) * 100
df_taxas['Taxa_15d'] = (df_taxas['Ganho_15d'] / df_taxas['Ganho_Total']) * 100
df_taxas['Taxa_30d'] = (df_taxas['Ganho_30d'] / df_taxas['Ganho_Total']) * 100

# Prepara dados
df_plot = df_taxas[['Taxa_7d', 'Taxa_15d', 'Taxa_30d']]
x = range(len(df_plot))
bar_width = 0.25
labels = df_plot.index
colors = ['#4caf50', '#ff9800', '#2196f3']

plt.figure(figsize=(12,6))

# Barras agrupadas
for i, coluna in enumerate(df_plot.columns):
    offset = [-bar_width, 0, bar_width][i]
    valores = df_plot[coluna].values
    posicoes = [xi + offset for xi in x]
    bars = plt.bar(posicoes, valores, bar_width, label=coluna.replace('Taxa_', '') + ' dias', color=colors[i])

    # Adiciona os valores nas barras
    for bar in bars:
        altura = bar.get_height()
        if altura > 0:
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + altura + 1,
                f'{altura:.1f}%',
                ha='center',
                va='bottom',
                fontsize=9,
                color='black'
            )

# Configurações finais do gráfico
plt.title('Taxas Relativas ao Ganho Total por Mês (7, 15, 30 dias)')
plt.xlabel('Mês de Ganho')
plt.ylabel('Taxa de Ganho (%)')
plt.xticks(ticks=x, labels=labels, rotation=45)
plt.legend(title='Prazo')
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Salvar imagem
img_path = "grafico_taxa_ganhos_lado_a_lado.png"
plt.savefig(img_path)
plt.close()

# Gera PowerPoint
ppt_path = r"C:\Users\Magda\Downloads\Taxas_de_Conversao_LADO_A_LADO.pptx"
prs = Presentation()
slide = prs.slides.add_slide(prs.slide_layouts[5])
slide.shapes.add_picture(img_path, Inches(1), Inches(1), width=Inches(8), height=Inches(5))

# Salva e remove imagem temporária
prs.save(ppt_path)
os.remove(img_path)

print(f"Apresentação salva com sucesso: {ppt_path}")


Apresentação salva com sucesso: C:\Users\Magda\Downloads\Taxas_de_Conversao_LADO_A_LADO.pptx


In [27]:
# Escolha o mês desejado (formato: YYYY-MM)
mes_alvo = "2024-02"

# Verifica se o mês está presente
if mes_alvo in df_taxas.index:
    total = df_taxas.loc[mes_alvo, 'Ganho_Total']
    ganhos30 = df_taxas.loc[mes_alvo, 'Ganho_30d']
    print(f"Para o mês {mes_alvo}:")
    print(f"- Ganhos totais: {int(total)}")
    print(f"- Ganhos em até 30 dias: {int(ganhos30)}")
else:
    print(f"Mês {mes_alvo} não encontrado na base.")


Para o mês 2024-02:
- Ganhos totais: 76
- Ganhos em até 30 dias: 62


In [29]:
print(ganhos_total)

AnoMes
2024-01     61
2024-02     76
2024-03    127
2024-04    173
2024-05    168
2024-06    206
2024-07    206
2024-08    239
2024-09    212
2024-10    241
2024-11    423
2024-12    303
2025-01    166
2025-02    243
2025-03    130
Name: Ganho_Total, dtype: int64


In [31]:
import pandas as pd
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches
import os

# Leitura do arquivo
arquivo = r"C:\Users\Magda\Downloads\Cópia de Planilha de Case para KPIs.xlsx"
df = pd.read_excel(arquivo, sheet_name='deal list')
df.columns = df.columns.str.strip()

# Converte datas
df['Data_Recebido'] = pd.to_datetime(df['Negócio - Data recebido'], errors='coerce')
df['Data_Ganho'] = pd.to_datetime(df['Negócio - Ganho em'], dayfirst=True, errors='coerce')

# Filtra ganhos válidos
df = df[
    df['Data_Recebido'].notna() &
    df['Data_Ganho'].notna() &
    (df['Negócio - Status'].str.lower() == 'ganho')
].copy()

# Calcula tempo de conversão e mês de ganho
df['Dias_Conversao'] = (df['Data_Ganho'] - df['Data_Recebido']).dt.days
df['AnoMes'] = df['Data_Ganho'].dt.to_period('M').astype(str)

# Define categorias
categorias = {
    'Shopping': df[df['Negócio - Criador shopping'].notna()],
    'Alocados': df[df['Negócio - [SDR] Intermediário'].notna()],
    'Diretos': df[df['Negócio - [SDR} Quem?'].str.lower() == 'automação']
}

# Função para criar gráfico e salvar imagem
def gerar_grafico_taxas_por_categoria(df_categoria, nome_categoria):
    ganhos_total = df_categoria.groupby('AnoMes').size().rename("Ganho_Total")
    ganhos_7 = df_categoria[df_categoria['Dias_Conversao'] <= 7].groupby('AnoMes').size().rename("Ganho_7d")
    ganhos_15 = df_categoria[df_categoria['Dias_Conversao'] <= 15].groupby('AnoMes').size().rename("Ganho_15d")
    ganhos_30 = df_categoria[df_categoria['Dias_Conversao'] <= 30].groupby('AnoMes').size().rename("Ganho_30d")

    df_taxas = pd.concat([ganhos_total, ganhos_7, ganhos_15, ganhos_30], axis=1).fillna(0)
    df_taxas['Taxa_7d'] = (df_taxas['Ganho_7d'] / df_taxas['Ganho_Total']) * 100
    df_taxas['Taxa_15d'] = (df_taxas['Ganho_15d'] / df_taxas['Ganho_Total']) * 100
    df_taxas['Taxa_30d'] = (df_taxas['Ganho_30d'] / df_taxas['Ganho_Total']) * 100

    df_plot = df_taxas[['Taxa_7d', 'Taxa_15d', 'Taxa_30d']]
    x = range(len(df_plot))
    bar_width = 0.25
    labels = df_plot.index
    colors = ['#4caf50', '#ff9800', '#2196f3']

    plt.figure(figsize=(12,6))
    for i, coluna in enumerate(df_plot.columns):
        offset = [-bar_width, 0, bar_width][i]
        valores = df_plot[coluna].values
        posicoes = [xi + offset for xi in x]
        bars = plt.bar(posicoes, valores, bar_width, label=coluna.replace('Taxa_', '') + ' dias', color=colors[i])
        for bar in bars:
            altura = bar.get_height()
            if altura > 0:
                plt.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + altura + 1,
                    f'{altura:.1f}%',
                    ha='center',
                    va='bottom',
                    fontsize=9,
                    color='black'
                )

    plt.title(f'Taxas Relativas ao Ganho Total por Mês ({nome_categoria})')
    plt.xlabel('Mês de Ganho')
    plt.ylabel('Taxa de Ganho (%)')
    plt.xticks(ticks=x, labels=labels, rotation=45)
    plt.legend(title='Prazo')
    plt.tight_layout()
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    img_path = f"grafico_taxa_{nome_categoria.lower()}.png"
    plt.savefig(img_path)
    plt.close()
    return img_path

# Cria apresentação e adiciona slides por categoria
ppt_path = r"C:\Users\Magda\Downloads\Taxas_por_categoria.pptx"
prs = Presentation()

for categoria, df_categoria in categorias.items():
    if not df_categoria.empty:
        img_path = gerar_grafico_taxas_por_categoria(df_categoria, categoria)
        slide = prs.slides.add_slide(prs.slide_layouts[5])
        slide.shapes.add_picture(img_path, Inches(1), Inches(1), width=Inches(8), height=Inches(5))
        os.remove(img_path)

# Salva apresentação
prs.save(ppt_path)
print(f"Apresentação criada com sucesso em: {ppt_path}")


Apresentação criada com sucesso em: C:\Users\Magda\Downloads\Taxas_por_categoria.pptx


In [33]:
import pandas as pd
import matplotlib.pyplot as plt

# Caminho do arquivo
arquivo = r"C:\Users\Magda\Downloads\Cópia de Planilha de Case para KPIs.xlsx"

# Carrega abas necessárias
df_deals = pd.read_excel(arquivo, sheet_name='deal list')
df_lojas = pd.read_excel(arquivo, sheet_name='Divisão Shopping')

# Remove espaços extras dos nomes das colunas
df_deals.columns = df_deals.columns.str.strip()
df_lojas.columns = df_lojas.columns.str.strip()

# Filtra apenas os registros com pré-busca realizada
df_prebusca = df_deals[df_deals['Negócio - Resultado pré-busca'].notna()].copy()

# Relaciona SDR com sua respectiva loja
df_merged = df_prebusca.merge(
    df_lojas[['Nome', 'Origem']],  # Nome = SDR, Origem = Loja
    left_on='Negócio - [SDR] Original',
    right_on='Nome',
    how='left'
)

# Preenche lojas não identificadas
df_merged['Origem'] = df_merged['Origem'].fillna('Desconhecida')

# Agrupa por loja (Origem)
prebusca_por_loja = df_merged.groupby('Origem').size().reset_index(name='Qtde_Prebusca')

# Ordena
prebusca_por_loja = prebusca_por_loja.sort_values(by='Qtde_Prebusca', ascending=False)

# Gráfico
plt.figure(figsize=(10,6))
bars = plt.barh(prebusca_por_loja['Origem'], prebusca_por_loja['Qtde_Prebusca'], color='#2196f3')
plt.xlabel('Quantidade de Pré-Buscas Realizadas')
plt.title('Uso da Pré-Busca por Loja')
plt.gca().invert_yaxis()

# Adiciona valores nas barras
for bar in bars:
    width = bar.get_width()
    plt.text(width + 1, bar.get_y() + bar.get_height()/2, str(int(width)), va='center')

plt.tight_layout()
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Salvar imagem
img_path = r"C:\Users\Magda\Downloads\prebusca_por_loja.png"
plt.savefig(img_path)
plt.close()

print(f"Gráfico salvo com sucesso em: {img_path}")


Gráfico salvo com sucesso em: C:\Users\Magda\Downloads\prebusca_por_loja.png


In [39]:
import pandas as pd
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches
import os

# Carrega os dados
arquivo = r"C:\Users\Magda\Downloads\Cópia de Planilha de Case para KPIs.xlsx"
df = pd.read_excel(arquivo, sheet_name='deal list')
df.columns = df.columns.str.strip()

# Filtra somente negócios perdidos
df_perdidos = df[df['Negócio - Status'].str.lower() == 'perdido'].copy()

# Classifica o tipo de lead
def classificar_lead(row):
    if pd.notna(row.get('Negócio - Criador shopping')):
        return 'Shopping'
    elif pd.notna(row.get('Negócio - [SDR] Intermediário')):
        return 'Alocados'
    elif str(row.get('Negócio - [SDR} Quem?')).strip().lower() == 'automação':
        return 'Diretos'
    else:
        return 'Outros'

df_perdidos['Tipo_Lead'] = df_perdidos.apply(classificar_lead, axis=1)
df_perdidos = df_perdidos[df_perdidos['Tipo_Lead'].isin(['Shopping', 'Alocados', 'Diretos'])]

# Agrupa por tipo e motivo
ranking = df_perdidos.groupby(['Tipo_Lead', 'Negócio - Motivo da perda']).size().reset_index(name='Qtd')
ranking = ranking[ranking['Qtd'] > 30]

# Cria apresentação
ppt_path = r"C:\Users\Magda\Downloads\Ranking_Motivos_MultiplosSlides.pptx"
prs = Presentation()

# Gera um slide para cada tipo
for tipo in ['Shopping', 'Alocados', 'Diretos']:
    dados = ranking[ranking['Tipo_Lead'] == tipo].sort_values(by='Qtd', ascending=True)

    if dados.empty:
        continue

    altura = max(6, len(dados) * 0.4)
    fig, ax = plt.subplots(figsize=(10, altura))
    bars = ax.barh(dados['Negócio - Motivo da perda'], dados['Qtd'], color='#2196f3')
    ax.set_title(f'Motivos da Perda - {tipo}', fontsize=14)
    ax.set_xlabel('Quantidade')
    ax.set_xlim(0, dados['Qtd'].max() * 1.2)

    for bar, qtd in zip(bars, dados['Qtd']):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, str(qtd), va='center', fontsize=9)

    ax.tick_params(axis='y', labelsize=9)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Salva imagem temporária
    img_path = f"grafico_motivos_{tipo}.png"
    plt.savefig(img_path, dpi=300)
    plt.close()

    # Adiciona slide e imagem
    slide = prs.slides.add_slide(prs.slide_layouts[5])
    slide.shapes.add_picture(img_path, Inches(0.5), Inches(0.5), width=Inches(9), height=Inches(6))
    os.remove(img_path)

# Salva apresentação
prs.save(ppt_path)
print(f"Apresentação salva com sucesso em: {ppt_path}")


Apresentação salva com sucesso em: C:\Users\Magda\Downloads\Ranking_Motivos_MultiplosSlides.pptx
